In [ ]:
# This code returns the fits the distribution of the cell's nucleus/membrane size to a lognormal distribution. 
# It takes in a directory containing all the masks of your nucleus/membrane stained samples (one cell type only)
# The below code is for a PI stained set of PBMC only assays, but can be used for any cell-line.

In [ ]:
# cell to measure the area of each PBMC nucleus detected by a set of masks, after removing small specks and large clumps, and 
# save each mask's info as a csv

import os
import subprocess
import torch

# INPUTS -----------------------------------------------------------------------------------------------------
# Directory containing the nuclear/membrane masks of all your stained assays
mask_directory = r"\Path\to\directory_of_nuclear_masks"

# path to your FIJI application (".exe" ending)
fiji_path = r"\Path\to\fiji-windows-x64.exe"

# path to "PBMC_mask_info_python.ijm" FIJI macro
PBMC_mask_info_macro_path = r"\Path\to\PBMC_mask_info_python.ijm"

# FUNCTIONS --------------------------------------------------------------------------------------------------
# PBMC info extraction function
def get_PBMC_info(image_path, fiji_path, macro_path):
    cmd = [fiji_path, "-macro", macro_path, image_path] 
    process = subprocess.run(cmd, check=False, capture_output=True, text=True)
    print("return code:", process.returncode)
    print("stdout:\n", process.stdout[:5000])
    print("stderr:\n", process.stderr[:5000])

# function to connect notebook to GPU
def connect_gpu():
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("Using:", device)

# MAIN SCRIPT ------------------------------------------------------------------------------------------------
valid_raw_data_exts = [".tif"]
connect_gpu()

for mask in os.listdir(mask_directory):

    if not any(mask.lower().endswith(ext) for ext in valid_raw_data_exts):
        continue
    
    mask_path = os.path.join(mask_directory, mask)
    mask_name, ext = os.path.splitext(os.path.basename(mask_path))

    get_PBMC_info(mask_path, fiji_path, PBMC_mask_info_macro_path)

print("done")

In [ ]:
# cell to combine these csvs into one dataset

import pandas as pd
import glob
import os

# INPUTS -----------------------------------------------------------------------------------------------------
output_folder_name = "total_dataset_outputs"

# FUNCTIONS --------------------------------------------------------------------------------------------------
# function to combine all the csvs in the mask directory
def combine_csvs(folder_path):
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    dfs = []
    
    for file in csv_files:
        df = pd.read_csv(file)
        df["source_file"] = os.path.basename(file)
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)    
    return combined_df

# MAIN SCRIPT ------------------------------------------------------------------------------------------------
output_folder_path = os.path.join(mask_directory, output_folder_name)
os.makedirs(output_folder_path, exist_ok=True)

combined_df = combine_csvs(mask_directory)
csv_name = "full_dataset_info.csv"
df_path = os.path.join(output_folder_path, csv_name)
combined_df.to_csv(df_path, index=False)

In [ ]:
# cell to plot PBMC nucleus areas as a histogram and fit a log-normal distribution to it (fits a normal distribution to the 
# log of the membrane distribution)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import lognorm, norm

hist_range = (30, 300)
ln_hist_range = (np.log(30), np.log(300))

df = pd.read_csv(df_path)
areas = df["Area"].dropna()

# finding ln of data to fit a normal distribution to
ln_areas = np.log(areas)

# calculate empirical mean, std and errors:
emp_ln_mean = ln_areas.mean()
emp_ln_std = ln_areas.std()
n = len(areas)
print(n)
error_emp_ln_mean = emp_ln_std/np.sqrt(n)
error_emp_ln_std = emp_ln_std/np.sqrt(2*(n-1))

# fit a normal distribution to data
ln_mean, ln_std = norm.fit(ln_areas)

emp_mean = areas.mean()
emp_std = areas.std()

# fit a lognormal distribution to data
shape, loc, scale = lognorm.fit(areas)
fitted_mean, var = lognorm.stats(shape, loc=loc, scale=scale, moments='mv')
fitted_std = np.sqrt(var)

distribution_df = pd.DataFrame({
    "Empirical Mean": [emp_mean],
    "Fitted Mean": [fitted_mean],
    "Empirical Std": [emp_std],
    "Fitted Std": [fitted_std],
    "Empirical ln Mean": [emp_ln_mean],
    "Absolute Error Empirical ln Mean": [error_emp_ln_mean],
    "Fitted ln Mean": [ln_mean],
    "Empirical ln STD": [emp_ln_std],
    "Absolute Error Empirical ln STD": [error_emp_ln_std],
    "Fitted ln Std": [ln_std]
})
distribution_csv_name = "distribution.csv"
distribution_csv_path = os.path.join(output_folder_path, distribution_csv_name)
distribution_df.to_csv(distribution_csv_path, index=False)

plt.figure()

# Plot histogram of areas
plt.hist(areas, bins=30, range=hist_range, density=True, label=f"measured values (μ={emp_mean:.4f}, σ={emp_std:.4f})")

# Plot fitted normal distribution
x = np.linspace(30, 300, 1000)
y = lognorm.pdf(x, shape, loc, scale)
plt.plot(x, y, label=f"Log-normal fit (μ = {fitted_mean:.4f}, σ = {fitted_std:.4f})")

plt.xlabel("nucleus area (square microns)")
plt.ylabel("Frequency density")
plt.title("Distribution of PBMC nucleus areas")
plt.legend(fontsize=8, borderpad=0.5, labelspacing=1.0, handletextpad=1.0)

figure_path = os.path.join(output_folder_path, "distribution.png")
plt.savefig(figure_path)
plt.show()
plt.close()

plt.figure()

# Plot histogram of areas
plt.hist(ln_areas, bins=30, range=ln_hist_range, density=True, label=f"measured values (μ = {emp_ln_mean:.4f} ± {error_emp_ln_mean:.4f}, \nσ = {emp_ln_std:.4f} ± {error_emp_ln_std:.4f})")

# Plot fitted normal distribution
ln_x = np.linspace(np.log(30), np.log(300), 1000)
ln_y = norm.pdf(ln_x, ln_mean, ln_std)
plt.plot(ln_x, ln_y, label=f"normal fit (μ = {ln_mean:.4f}, σ = {ln_std:.4f})")

plt.xlabel("log of nucleus area (square microns)")
plt.ylabel("Frequency density")
plt.title("Distribution of PBMC nucleus areas")
plt.legend(fontsize=8, borderpad=0.5, labelspacing=1.0, handletextpad=1.0)  
plt.ylim(0, 1.4)

ln_figure_path = os.path.join(output_folder_path, "ln_distribution.png")
plt.savefig(ln_figure_path)
plt.show()
plt.close()